# 02 — TimesFM 2.5 pipeline with exogenous covariates

A variant of [01_pipeline_timesfm.ipynb](01_pipeline_timesfm.ipynb) that incorporates covariates:
- **Dynamic**: annual share of enrollment by field of knowledge (institution).
- **Static**: subject profile of the publisher's catalog (shares by area).

Four scenarios are evaluated:

| Scenario | Dynamic cov. | Static cov. |
|---|---|---|
| A | — | — |
| B | Yes | — |
| C | — | Yes |
| D | Yes | Yes |

Each series automatically picks the scenario with the lowest MASE.

**Environment:** `.venv` (TimesFM `[xreg]`).

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_parquet, filter_period, build_series, input_matrix
from src.metrics import all_metrics

PARQUET_PATH = REPO_ROOT / 'data' / 'anonymized_series.parquet'
COV_DYN_PATH = REPO_ROOT / 'data' / 'dynamic_covariates.parquet'
COV_STAT_PATH = REPO_ROOT / 'data' / 'static_covariates.parquet'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'timesfm_cov'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 12

## 2. Loading series and building covariates

The covariates already come anonymized and summarized:
- `dynamic_covariates.parquet`: one row per (institution, year) with `prop_area_1` … `prop_area_K`.
- `static_covariates.parquet`: one row per publisher with `kbart_area_1` … `kbart_area_K`.

In [ ]:
df = load_parquet(PARQUET_PATH)
df = filter_period(df, 2020, 2024)
series = build_series(df, min_months=24)
train_list, test_list, ids = input_matrix(series, horizon=HORIZON)

cov_dyn = pd.read_parquet(COV_DYN_PATH)    # institution, year, prop_area_*
cov_stat = pd.read_parquet(COV_STAT_PATH)  # publisher, kbart_area_*

In [ ]:
def build_cov_arrays(ids_, series_, cov_dyn_, cov_stat_):
    cols_dyn = [c for c in cov_dyn_.columns if c.startswith('prop_area_')]
    cols_stat = [c for c in cov_stat_.columns if c.startswith('kbart_area_')]
    dyn = []
    sta = []
    for sid in ids_:
        s = series_[sid]
        # Dynamic: share for the corresponding year, repeated per month
        years = s.dates.year
        dyn_rows = cov_dyn_.set_index(['institution', 'year']).loc[s.institution]
        dyn_monthly = np.stack([dyn_rows.loc[y, cols_dyn].values for y in years])
        dyn.append(dyn_monthly.astype(np.float32))
        # Static: one row per publisher
        stat_row = cov_stat_.set_index('publisher').loc[s.publisher, cols_stat].values
        sta.append(stat_row.astype(np.float32))
    return dyn, np.stack(sta)

dyn_covs, static_covs = build_cov_arrays(ids, series, cov_dyn, cov_stat)

## 3. Loading the model

`forecast_with_covariates` requires `return_backcast=True`.

In [ ]:
from timesfm import TimesFM_2p5_200M_torch, ForecastConfig

model = TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
model.compile(ForecastConfig(
    max_context=1024,
    max_horizon=256,
    normalize_inputs=True,
    use_continuous_quantile_head=True,
    fix_quantile_crossing=True,
    return_backcast=True,
))

## 4. Four scenarios

In [ ]:
# Scenario A: no covariates
pf_A, _ = model.forecast(horizon=HORIZON, inputs=train_list)
pred_A = np.clip(pf_A[:, :HORIZON], 0, None)

# Scenario B: dynamic only
pf_B, _ = model.forecast_with_covariates(
    horizon=HORIZON,
    inputs=train_list,
    dynamic_numerical_covariates=dyn_covs,
)
pred_B = np.clip(pf_B[:, :HORIZON], 0, None)

# Scenario C: static only
pf_C, _ = model.forecast_with_covariates(
    horizon=HORIZON,
    inputs=train_list,
    static_numerical_covariates=static_covs,
)
pred_C = np.clip(pf_C[:, :HORIZON], 0, None)

# Scenario D: both
pf_D, _ = model.forecast_with_covariates(
    horizon=HORIZON,
    inputs=train_list,
    dynamic_numerical_covariates=dyn_covs,
    static_numerical_covariates=static_covs,
)
pred_D = np.clip(pf_D[:, :HORIZON], 0, None)

## 5. Per-scenario evaluation + per-series winner selection

In [ ]:
scenarios = {'A': pred_A, 'B': pred_B, 'C': pred_C, 'D': pred_D}
rows = []
for i, sid in enumerate(ids):
    y_true = test_list[i]
    train = train_list[i]
    for name, preds in scenarios.items():
        m = all_metrics(y_true, preds[i], train)
        rows.append({'series_id': sid, 'scenario': name, **m})

df_scn = pd.DataFrame(rows)
df_scn.to_csv(OUTPUT_DIR / 'scenario_metrics.csv', index=False)

# Winner per series (lowest MASE)
winners = df_scn.loc[df_scn.groupby('series_id')['MASE'].idxmin(), ['series_id', 'scenario', 'MASE']]
winners.columns = ['series_id', 'winning_scenario', 'best_MASE']
winners.to_csv(OUTPUT_DIR / 'winning_scenario.csv', index=False)
winners['winning_scenario'].value_counts()

## 6. Covariate impact (D vs A)

In [ ]:
pivot = df_scn.pivot(index='series_id', columns='scenario', values='MASE')
pivot['delta_D_A'] = pivot['D'] - pivot['A']
pivot['improvement_pct'] = pivot['delta_D_A'] / pivot['A'] * 100
pivot.to_csv(OUTPUT_DIR / 'impact_covariates.csv')
pivot[['A', 'D', 'delta_D_A', 'improvement_pct']].describe()

## 7. Export forecasts of the winning scenario (for comparison)

For each series the scenario with the lowest MASE is taken as the TimesFM-with-covariates forecast, exported as `predictions_timesfm_cov.csv` so notebook 06 can consume it.

In [ ]:
scenario_arrays = {'A': pred_A, 'B': pred_B, 'C': pred_C, 'D': pred_D}
winning_map = winners.set_index('series_id')['winning_scenario'].to_dict()

pred_rows = []
for i, sid in enumerate(ids):
    win = winning_map[sid]
    best_pred = scenario_arrays[win][i]
    for h in range(HORIZON):
        pred_rows.append({
            'series_id': sid,
            'horizon_month': h + 1,
            'y_true': float(test_list[i][h]),
            'pred_timesfm_cov': float(best_pred[h]),
        })
pd.DataFrame(pred_rows).to_csv(OUTPUT_DIR / 'predictions_timesfm_cov.csv', index=False)